In [ ]:
import os
import numpy as np
import pandas as pd
import mne
import mne_bids
from pathlib import Path

# -----------------------------
# Dataset information
# -----------------------------
bids_dir = r"E:\IEEG-FMRI Dataset"

subjects = mne_bids.get_entity_vals(bids_dir, "subject")

session = "iemu"
datatype = "ieeg"
task = "film"
acquisition = "clinical"

preprocessed_data = {}
summary = []

# -----------------------------
# Loop through all subjects
# -----------------------------
for subject in subjects:

    print(f"\nProcessing subject {subject}")

    # Skip subjects without iEEG session
    iemu_folder = Path(bids_dir) / f"sub-{subject}" / "ses-iemu"

    if not iemu_folder.exists():
        print(f"Skipping subject {subject}: no iEEG session")
        continue

    # ------------------------------------
    # Read channels.tsv
    # ------------------------------------
    channels_path = mne_bids.BIDSPath(
        subject=subject,
        session=session,
        datatype=datatype,
        task=task,
        acquisition=acquisition,
        suffix="channels",
        extension=".tsv",
        root=bids_dir,
    )

    # print(subject)
    # print(channels_path)
    # print(channels_path.match())

    channels = pd.read_csv(channels_path.match()[0], sep="\t")

    # ------------------------------------
    # Read BrainVision recording
    # ------------------------------------
    data_path = mne_bids.BIDSPath(
        subject=subject,
        session=session,
        datatype=datatype,
        task=task,
        acquisition=acquisition,
        suffix="ieeg",
        extension=".vhdr",
        root=bids_dir,
    )

    raw = mne.io.read_raw_brainvision(
        str(data_path.match()[0]),
        preload=True,
        verbose=False,
    )

    # ------------------------------------
    # Keep only EEG / ECoG / SEEG channels
    # ------------------------------------
    raw.set_channel_types(
        {
            ch_name: str(ch_type).lower()
            if str(ch_type).lower() in ["ecog", "seeg", "eeg"]
            else "misc"
            for ch_name, ch_type in zip(
                raw.ch_names,
                channels["type"].values
            )
        }
    )

    raw.drop_channels(
        [
            ch
            for ch, typ in zip(raw.ch_names, raw.get_channel_types())
            if typ == "misc"
        ]
    )

    # ------------------------------------
    # Remove bad/noisy channels
    # ------------------------------------
    bad_channels = channels.loc[
        channels["status"] == "bad",
        "name"
    ].tolist()

    bad_channels = [ch for ch in bad_channels if ch in raw.ch_names]

    raw.drop_channels(bad_channels)

    # ------------------------------------
    # Remove 50 Hz line noise and its harmonics
    # ------------------------------------
    raw.notch_filter(freqs=np.arange(50, 251, 50))

    # ------------------------------------
    # Common Average Reference
    # ------------------------------------
    raw_car, _ = mne.set_eeg_reference(raw.copy(), "average")

    # ------------------------------------
    # Store
    # ------------------------------------
    preprocessed_data[subject] = raw_car

    summary.append({
    "Subject": subject,
    "Sampling_Frequency": raw_car.info["sfreq"],
    "Channels_Remaining": len(raw_car.ch_names)
    })

    print(f"Sampling frequency: {raw_car.info['sfreq']} Hz")
    print(f"Remaining channels: {len(raw_car.ch_names)}")

summary_df = pd.DataFrame(summary)
summary_df


Processing subject 01


C:\Users\muqadasah\AppData\Local\Temp\ipykernel_6828\3625936425.py:80: RuntimeWarning: The unit for channel(s) AH+, ECG+, EMG+, MKR1+, MKR2+, ORB+, abdo+, thor+ has changed from V to NA.
  raw.set_channel_types(


Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 13517 samples (6.600 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.5s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    2.3s


sEEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('sEEG',) reference.
Sampling frequency: 2048.0 Hz
Remaining channels: 101

Processing subject 02
Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 3381 samples (6.604 s)



...

C:\Users\muqadasah\AppData\Local\Temp\ipykernel_6828\3625936425.py:80: RuntimeWarning: The unit for channel(s) AH+, ECG+, EMG+, MKR1+, MKR2+, Orb+ has changed from V to NA.
  raw.set_channel_types(


- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 3381 samples (6.604 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


ECoG channel type selected for re-referencing
Applying average reference.
Applying a custom ('ECoG',) reference.
Sampling frequency: 512.0 Hz
Remaining channels: 70


,Subject,Sampling_Frequency,Channels_Remaining
0,01,2048.0,101
1,02,512.0,59
2,03,512.0,101
3,05,2048.0,73
4,06,512.0,102
5,07,512.0,55
6,09,2048.0,69
7,10,512.0,90
8,12,2048.0,64
9,13,512.0,94
